# Data Preprocessing Objective

The objective of this notebook is to prepare the customer churn dataset for machine learning by cleaning the data, transforming features, encoding categorical variables, and creating training and testing datasets.

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline 

In [2]:
df=pd.read_csv('Data/customer.csv')
               

In [3]:
df.head()

,user_id,signup_date,plan_type,monthly_fee,avg_weekly_usage_hours,support_tickets,payment_failures,tenure_months,last_login_days_ago,churn
0,1,2023-04-15,Premium,699,1.1,4,1,8,14,Yes
1,2,2023-08-27,Premium,699,2.6,6,0,35,1,Yes
2,3,2023-10-12,Premium,699,14.3,8,3,2,14,Yes
3,4,2023-12-11,Basic,199,17.6,5,2,11,9,Yes
4,5,2023-02-14,Basic,199,9.8,5,2,6,38,Yes


In [4]:
df.shape

(2800, 10)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2800 entries, 0 to 2799
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   user_id                 2800 non-null   int64  
 1   signup_date             2800 non-null   object 
 2   plan_type               2800 non-null   object 
 3   monthly_fee             2800 non-null   int64  
 4   avg_weekly_usage_hours  2800 non-null   float64
 5   support_tickets         2800 non-null   int64  
 6   payment_failures        2800 non-null   int64  
 7   tenure_months           2800 non-null   int64  
 8   last_login_days_ago     2800 non-null   int64  
 9   churn                   2800 non-null   object 
dtypes: float64(1), int64(6), object(3)
memory usage: 218.9+ KB


In [6]:
# Checking Missing values 
df.isnull().sum()

user_id                   0
signup_date               0
plan_type                 0
monthly_fee               0
avg_weekly_usage_hours    0
support_tickets           0
payment_failures          0
tenure_months             0
last_login_days_ago       0
churn                     0
dtype: int64

### Observation

No missing values were found, so no imputation was required.

In [7]:
# Checking Duplicates 
df.duplicated().sum()

np.int64(0)

### Observation

No duplicate records were found, so no duplicate rows were removed.

In [8]:
df['signup_date'] = pd.to_datetime(df['signup_date'])

### Observation

The `signup_date` column was converted from object to datetime format to enable feature extraction.

### Feature Engineering

In [9]:
# Extract from date column 
df['signup_year'] = df['signup_date'].dt.year
df['signup_month'] = df['signup_date'].dt.month
df['signup_day'] = df['signup_date'].dt.day

#dropping the original 'signup_date'
df.drop(columns='signup_date', inplace=True)

In [10]:
# dropping unnecessary columns 
df.drop(columns='user_id', inplace=True)

### Separate Features and Target

In [11]:
X = df.drop('churn', axis=1)
Y = df['churn']

In [12]:
# identifying Numerical and Categorical values 
numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns

print(numerical_columns)


Index(['monthly_fee', 'avg_weekly_usage_hours', 'support_tickets',
       'payment_failures', 'tenure_months', 'last_login_days_ago'],
      dtype='object')


In [13]:
categorical_columns = df.select_dtypes(include=['object', 'category']).columns

print(categorical_columns)

Index(['plan_type', 'churn'], dtype='object')


### Encoding Categorical Features 

In [14]:
X_train,X_test,y_train,y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

In [15]:
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(2240, 10)
(560, 10)
(2240,)
(560,)


In [16]:
print(X.columns)

Index(['plan_type', 'monthly_fee', 'avg_weekly_usage_hours', 'support_tickets',
       'payment_failures', 'tenure_months', 'last_login_days_ago',
       'signup_year', 'signup_month', 'signup_day'],
      dtype='object')


### One Hot Encoding for Categorical Values 

In [17]:
trf1=ColumnTransformer(
    transformers=[
    ('ohe_plan_type', 
     OneHotEncoder(drop='first',sparse_output=False , handle_unknown='ignore') ,
     [0]
    )
    ] ,
    remainder='passthrough'
    )

### Observation

- The categorical feature `plan_type` was transformed using One-Hot Encoding.
- New binary columns were created to represent each plan category.
- The first category was dropped (`drop='first'`) to avoid redundant features and reduce multicollinearity.
- The remaining numerical features were passed through without modification.

### Feature Scaling using Standard Scaler 

In [18]:
from sklearn.preprocessing import StandardScaler
trf2 = ColumnTransformer(
    transformers=[
        (
            'scale',
            StandardScaler(),
            [2,3,4,5,6,7,8,9,10]
        )
    ],
    remainder='passthrough'
)

### Observation

- Numerical features were standardized using `StandardScaler`.
- Feature scaling ensures that all numerical variables are on a similar scale, preventing features with larger values from dominating the learning process.
- The categorical features generated by One-Hot Encoding were not scaled because they already contain binary values (0 and 1).

### Creating the Pipeline 

In [19]:
pipe = Pipeline([
    ('trf1', trf1),
    ('trf2', trf2)
])

### Fitting the pipeline

In [20]:
pipe.fit(X_train, y_train)

X_train_transformed = pipe.transform(X_train)
X_test_transformed = pipe.transform(X_test)

### Observation

- The preprocessing pipeline was fitted using the training dataset.
- During this step, the One-Hot Encoder learned the categories present in the categorical feature, and the StandardScaler calculated the mean and standard deviation of the numerical features.
- These learned parameters will later be used to transform both the training and testing datasets consistently.

In [21]:
X_train_transformed.shape

(2240, 11)

In [22]:
column_names = [
    'monthly_fee',
    'avg_weekly_usage_hours',
    'support_tickets',
    'payment_failures',
    'tenure_months',
    'last_login_days_ago',
    'signup_year',
    'signup_month',
    'signup_day',
    'plan_type_Premium',
    'plan_type_Standard'
]

X_train_df = pd.DataFrame(
    X_train_transformed,
    columns=column_names
)

X_train_df.head()

,monthly_fee,avg_weekly_usage_hours,support_tickets,payment_failures,tenure_months,last_login_days_ago,signup_year,signup_month,signup_day,plan_type_Premium,plan_type_Standard
0,-0.165518,0.675487,0.416075,0.892117,-1.593039,0.334827,-1.044750,0.134932,-0.433288,0.0,1.0
1,-1.141203,0.563056,-0.728429,1.480903,1.187289,-0.057791,0.957166,0.427807,0.484721,0.0,0.0
2,-0.165518,-0.364496,-0.346928,-1.463029,-1.401293,1.680945,0.957166,-0.743694,-0.662791,0.0,1.0
3,1.298010,-1.446640,0.797576,0.892117,-0.250812,1.288327,0.957166,0.427807,-1.466049,1.0,0.0
4,1.298010,1.040886,-0.728429,-0.285456,0.995543,-1.684350,0.957166,1.306433,-1.121795,1.0,0.0


### Observation 

### Observation

- The preprocessing pipeline successfully transformed the training data.
- All numerical features were standardized using `StandardScaler`, resulting in values centered around 0 with a standard deviation close to 1.
- The categorical feature `plan_type` was converted into binary columns using One-Hot Encoding.
- The transformed dataset is now entirely numerical and ready to be used for training machine learning models.